## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

True

In [2]:
choose_model_0="qwen2.5:7b"
choose_model_1="llama3.2:latest"
choose_model_3="mapler/gpt2:latest"
choose_model_2="gemma3:270m"

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [3]:

#
# Because I want to use Ollama in local
from openai import AsyncOpenAI
from agents import  OpenAIChatCompletionsModel, set_default_openai_client
from agents import (set_tracing_disabled, set_default_openai_client, set_default_openai_api)
set_tracing_disabled(True)

client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  
)

set_default_openai_client(client, use_for_tracing=False)
set_default_openai_api("chat_completions")   # Ollama ne gère pas l'API "responses"




In [4]:

# Make an agent with name, instructions, model
agent = Agent(
    name="Jokester", 
    instructions="You are a joke teller", 
    model=choose_model_0
)

In [5]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [6]:
# Here is the final output

print(result.final_output)

Sure! Here's a joke about Autonomous AI Agents:

Why did the autonomous AI agent refuse to cross the road?

Because it already had a million tabs open and didn't want to increase its tab-closure time!


In [7]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Sure! Here's a joke about Autonomous AI Agents:\n\nWhy did the autonomous AI agent refuse to cross the road?\n\nBecause it already had a million tabs open and didn't want to increase its tab-closure time!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-931'}}]

## Adding Observability with a trace

In [8]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent break into a bakery?

Because it needed to test its boundary conditions!


## Now go and look at the trace

https://platform.openai.com/traces

In [9]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure, here are five AI agent jokes for you:

1. **Why did the AI agent refuse a cup of coffee?**
   Because it was already trained on too many data points!

2. **Why did the AI agent break up with the machine learning model?**
   It said, "I just don't have the right fit anymore."

3. **How do AI agents deal with their feelings?**
   They use sentiment analysis to keep things positive!

4. **Why did the AI agent go to therapy?**
   It had recurrent model issues.

5. **Why did the AI agent refuse to answer the phone?**
  Because it was afraid of getting caught in loops!

I hope you found these jokes amusing! Do you have a favorite or would you like more?

## Part 2: Adding a tool

In [11]:
pushover_user = os.getenv("PUSH_OVER_USER_KEY")
pushover_token = os.getenv("PUSH_OVER_APP_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [12]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [13]:
push("HEY!!")

Push: HEY!!


In [14]:
push

<function __main__.push(message)>

In [15]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [16]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002A03DA8C3B0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [17]:
push_tool.description

'Send the given message to the user as a push notification'

In [18]:

notifier = Agent(name="Notifier", model=choose_model_0, instructions="You notify the user upon request", tools=[push_tool])

In [20]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


The notification has been sent successfully informing you that the pizza is here!


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [22]:
agent = Agent(name="Assistant", model=choose_model_0)

In [24]:
response = await Runner.run(agent, "Hi there. My name is Mervy.")
print(response.final_output)

Hello Mervy! Nice to meet you. How can I assist you today? Do you have any specific questions or topics you'd like to discuss?


In [25]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

You didn't provide a name, so I don't know your name. You can tell me your name, and I'd be happy to help you with anything!


## Memory approach 1 - just manually pass in the list of dicts

In [27]:
response = await Runner.run(agent, "Hi there. My name is Mervy.")
print(response.final_output)

Hello Mervy! Nice to meet you. How can I assist you today? Feel free to ask about any topics or discuss any interests you have.


In [28]:
response.to_input_list()

[{'content': 'Hi there. My name is Mervy.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hello Mervy! Nice to meet you. How can I assist you today? Feel free to ask about any topics or discuss any interests you have.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-145'}}]

In [29]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Mervy.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hello Mervy! Nice to meet you. How can I assist you today? Feel free to ask about any topics or discuss any interests you have.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-145'}},
 {'role': 'user', 'content': "What's my name?"}]

In [30]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Mervy. How about we talk about something you're interested in or need help with today?


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# SQLiteSession de "agents"
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346", "memory.db")

In [34]:
response = await Runner.run(agent, "Hi there. My name is Mervy.", session=session)
print(response.final_output)

Hello Mervy! Nice to meet you. How can I assist you today? Whether you have questions, need some information, or just want to talk about something, feel free to let me know.


In [35]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Mervy! How about we proceed from there? Do you have any specific questions or topics you'd like to discuss?


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>